# News Sources — Dev Playground

Test every news source in isolation and run the full aggregator.

In [ ]:
import asyncio, json, os, sys
from pathlib import Path
from datetime import datetime, timezone

# make alphaforge_anton_news importable from workspace root
ROOT = Path.cwd().parents[1]  # news/notebooks -> news -> repo root
NEWS_SRC = ROOT / 'news' / 'src'
if str(NEWS_SRC) not in sys.path:
    sys.path.insert(0, str(NEWS_SRC))

from alphaforge_anton_news import NewsAggregator, NewsItem, SourceHealth
from alphaforge_anton_news.sources import build_all_sources

def pp(obj):
    print(json.dumps(obj, indent=2, default=str))

def show_items(items: list[NewsItem], n: int = 5):
    for i in items[:n]:
        print(f"  [{i.source_slug:20}] {i.published_at.strftime('%m-%d %H:%M')}  {i.headline[:80]}")
    if len(items) > n:
        print(f"  ... +{len(items) - n} more")

print('alphaforge_anton_news loaded OK')

## 1. All sources registered

In [ ]:
sources = build_all_sources()
print(f'Total sources: {len(sources)}')
print()
for s in sources:
    key_needed = f'  [key: {s.env_key}]' if s.requires_api_key else ''
    key_set    = ' ✓' if (s.env_key and os.getenv(s.env_key)) else (' ✗' if s.requires_api_key else '')
    print(f'  {s.name:25}  category={s.category:6}{key_needed}{key_set}')

## 2. Health check — all sources

In [ ]:
agg = NewsAggregator(sources=build_all_sources())

healths: list[SourceHealth] = await agg.health()
print(f'{'Source':<26} {'Available':<12} {'Error'}')
print('-' * 70)
for h in healths:
    err = (h.last_error or '')[:50]
    avail = '✓' if h.available else '✗'
    print(f'  {h.name:<24} {avail:<12} {err}')

## 3. RSS sources only

In [ ]:
from alphaforge_anton_news.sources.rss import build_rss_sources

rss_agg = NewsAggregator(sources=build_rss_sources())
items = await rss_agg.search('market rally Nifty', limit=30)
print(f'RSS results: {len(items)}')
show_items(items, n=10)

## 4. Single RSS feed — Moneycontrol

In [ ]:
from alphaforge_anton_news.sources.rss import build_rss_sources

mc = next(s for s in build_rss_sources() if s.name == 'moneycontrol-rss')
items = await mc.search('Sensex', limit=5)
print(f'Moneycontrol results: {len(items)}')
for i in items:
    print(f'  {i.published_at.strftime("%m-%d %H:%M")}  {i.headline}')
    print(f'    {i.url}')

## 5. NSE announcements

In [ ]:
from alphaforge_anton_news.sources.nse_announcements import NseAnnouncementsSource

nse = NseAnnouncementsSource()
h = await nse.health()
print('NSE health:', 'available' if h.available else f'unavailable — {h.error}')

items = await nse.search('', symbols=['RELIANCE', 'TCS'], limit=10)
print(f'NSE results: {len(items)}')
show_items(items)

## 6. BSE announcements

In [ ]:
from alphaforge_anton_news.sources.bse_announcements import BseAnnouncementsSource

bse = BseAnnouncementsSource()
h = await bse.health()
print('BSE health:', 'available' if h.available else f'unavailable — {h.error}')

items = await bse.search('results', limit=10)
print(f'BSE results: {len(items)}')
show_items(items)

## 7. yfinance (per-symbol news)

In [ ]:
from alphaforge_anton_news.sources.yfinance_news import YFinanceSource

yf = YFinanceSource()
h = await yf.health()
print('yfinance health:', 'available' if h.available else f'unavailable — {h.last_error}')

items = await yf.search('', symbols=['INFY', 'HDFC', 'RELIANCE'], limit=15)
print(f'yfinance results: {len(items)}')
show_items(items)

## 8. Reddit India (requires REDDIT_CLIENT_ID + REDDIT_CLIENT_SECRET)

In [ ]:
from alphaforge_anton_news.sources.reddit import RedditSource

reddit = RedditSource()
h = await reddit.health()
print('Reddit health:', 'available' if h.available else f'unavailable — {h.last_error}')

if h.available:
    items = await reddit.search('Nifty breakout', limit=10)
    print(f'Reddit results: {len(items)}')
    show_items(items)
else:
    print('Set REDDIT_CLIENT_ID + REDDIT_CLIENT_SECRET in .env.cred.local to enable')

## 9. API-keyed sources (Newsdata / GNews / Tavily / Brave)

In [ ]:
from alphaforge_anton_news.sources.newsdata import NewsdataSource
from alphaforge_anton_news.sources.gnews import GnewsSource
from alphaforge_anton_news.sources.tavily import TavilySource
from alphaforge_anton_news.sources.brave import BraveSource

api_sources = [
    NewsdataSource(),
    GnewsSource(),
    TavilySource(),
    BraveSource(),
]

QUERY = 'Nifty 50 outlook'

for src in api_sources:
    h = await src.health()
    if not h.available:
        print(f'{src.name:20} ✗  {h.last_error or "no key"}')
        continue
    items = await src.search(QUERY, limit=5)
    print(f'{src.name:20} ✓  {len(items)} results')
    show_items(items, n=3)
    print()

## 10. Full aggregator — fan-out search

In [ ]:
QUERY   = 'Reliance Industries results'
SYMBOLS = ['RELIANCE']
LIMIT   = 30

items = await agg.search(QUERY, symbols=SYMBOLS, limit=LIMIT)
print(f'Aggregated results: {len(items)}')
print()
show_items(items, n=20)

## 11. Deduplication check

In [ ]:
from alphaforge_anton_news.dedup import deduplicate

# Run two overlapping queries and see how many dups are removed
batch1 = await agg.search('Nifty', limit=20)
batch2 = await agg.search('Nifty Bank', limit=20)
combined = batch1 + batch2
unique   = deduplicate(combined)

print(f'batch1={len(batch1)}  batch2={len(batch2)}  combined={len(combined)}  after_dedup={len(unique)}')
print(f'Dups removed: {len(combined) - len(unique)}')

## 12. Filter by `since` datetime

In [ ]:
from datetime import timedelta

since = datetime.now(timezone.utc) - timedelta(hours=6)
items = await agg.search('Nifty', since=since, limit=20)
print(f'Results from last 6 hours: {len(items)}')
show_items(items)

## 13. Backend API (via HTTP — needs server running on :8000)

In [ ]:
import httpx

BASE        = 'http://localhost:8000/api/v1'
AF_USERNAME = os.getenv('AF_USERNAME', 'admin')
AF_PASSWORD = os.getenv('AF_PASSWORD', 'alphaforge-anton-dev')

async def get_token():
    async with httpx.AsyncClient(base_url=BASE) as c:
        r = await c.post('/auth/token', data={'username': AF_USERNAME, 'password': AF_PASSWORD})
        r.raise_for_status()
        return r.json()['access_token']

try:
    token = await asyncio.wait_for(get_token(), timeout=5)
    print('Auth OK')
except Exception as e:
    token = None
    print(f'Server not reachable ({e}) — skipping HTTP tests')

In [ ]:
if token:
    headers = {'Authorization': f'Bearer {token}'}
    async with httpx.AsyncClient(base_url=BASE, headers=headers, timeout=30) as c:
        r = await c.get('/news/search', params={'q': 'Nifty', 'limit': 10})
        items_json = r.json()
    print(f'GET /news/search → {r.status_code}  ({len(items_json)} items)')
    for item in items_json[:5]:
        print(f"  [{item['source_slug']:20}] {item['headline'][:70]}")

In [ ]:
if token:
    headers = {'Authorization': f'Bearer {token}'}
    async with httpx.AsyncClient(base_url=BASE, headers=headers, timeout=30) as c:
        r = await c.get('/news/sources')
    print(f'GET /news/sources → {r.status_code}')
    for s in r.json():
        avail = '✓' if s['available'] else '✗'
        print(f"  {avail} {s['source_name']:26} {s.get('error') or ''}")